In [1]:
import sys
import time
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

# Project root
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_all_train_data
from src.normalization import (
    normalize_text,
    normalize_business_name,
    normalize_business_address,
    normalize_country,
    normalize_dataframe,
    detect_script,
)

tqdm.pandas()

c:\Users\dayan\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("Loading training data...")

start_time = time.time()

train_data = load_all_train_data()

elapsed = time.time() - start_time

print(f"\n✓ Training data loaded in {elapsed:.2f} seconds")

for name, df in train_data.items():
    print(f"{name:15s}: {len(df):,} rows × {len(df.columns)} columns")

Loading training data...

========== LOADING TRAIN DATA ==========


Train files:   0%|          | 0/4 [00:00<?, ?file/s]


Loading: train_source1.tsv


Train files:  25%|██▌       | 1/4 [00:05<00:17,  5.75s/file]

  ✓ Loaded 2,206,821 rows × 4 columns

Loading: train_source2.tsv


...

  ✓ Loaded 5,034,616 rows × 4 columns

Loading: train_source3.tsv


Train files:  75%|███████▌  | 3/4 [00:42<00:15, 15.78s/file]

  ✓ Loaded 5,285,603 rows × 4 columns

Loading: train_ground_truth.tsv


Train files: 100%|██████████| 4/4 [00:47<00:00, 11.90s/file]

  ✓ Loaded 2,206,821 rows × 2 columns

✓ Training data loaded in 47.63 seconds
source1        : 2,206,821 rows × 4 columns
source2        : 5,034,616 rows × 4 columns
source3        : 5,285,603 rows × 4 columns
ground_truth   : 2,206,821 rows × 2 columns


In [3]:
examples = [
    "Samsung Galaxy S24",
    "SAMSUNG   GALAXY   S24",
    "Samsung-Galaxy S24",
    "राम मार्केटिंग प्राइवेट लिमिटेड",
    "आदित्य प्रॉपर्टीज एलएलपी",
    "SCI Ptit Àmicale",
]

for text in examples:

    print(f"Original   : {text}")
    print(f"Normalized : {normalize_text(text)}")
    print(f"Script     : {detect_script(text)}")
    print("-" * 70)

Original   : Samsung Galaxy S24
Normalized : samsung galaxy s24
Script     : Latin
----------------------------------------------------------------------
Original   : SAMSUNG   GALAXY   S24
Normalized : samsung galaxy s24
Script     : Latin
----------------------------------------------------------------------
Original   : Samsung-Galaxy S24
Normalized : samsung galaxy s24
Script     : Latin
----------------------------------------------------------------------
Original   : राम मार्केटिंग प्राइवेट लिमिटेड
Normalized : र म म र क ट ग प र इव ट ल म ट ड
Script     : Devanagari
----------------------------------------------------------------------
Original   : आदित्य प्रॉपर्टीज एलएलपी
Normalized : आद त य प र पर ट ज एलएलप
Script     : Devanagari
----------------------------------------------------------------------
Original   : SCI Ptit Àmicale
Normalized : sci ptit àmicale
Script     : Latin
----------------------------------------------------------------------


In [4]:
SAMPLE_SIZE = 10_000

sample = (
    train_data["source1"]
    .head(SAMPLE_SIZE)
    .copy()
)

print(f"Sample size: {len(sample):,}")

Sample size: 10,000


In [5]:
start_time = time.time()

sample["business_name_normalized"] = (
    sample["business_name"]
    .progress_apply(normalize_business_name)
)

elapsed = time.time() - start_time

print(f"\n✓ Name normalization completed in {elapsed:.2f} seconds")

100%|██████████| 10000/10000 [00:00<00:00, 72645.37it/s]


✓ Name normalization completed in 0.17 seconds


In [6]:
start_time = time.time()

sample["business_address_normalized"] = (
    sample["business_address"]
    .progress_apply(normalize_business_address)
)

elapsed = time.time() - start_time

print(f"\n✓ Address normalization completed in {elapsed:.2f} seconds")

100%|██████████| 10000/10000 [00:00<00:00, 63648.81it/s]


✓ Address normalization completed in 0.16 seconds


In [7]:
start_time = time.time()

sample["country_normalized"] = (
    sample["country"]
    .progress_apply(normalize_country)
)

elapsed = time.time() - start_time

print(f"\n✓ Country normalization completed in {elapsed:.2f} seconds")

100%|██████████| 10000/10000 [00:00<00:00, 233564.47it/s]


✓ Country normalization completed in 0.05 seconds


In [8]:
display(
    sample[
        [
            "entity_id",
            "business_name",
            "business_name_normalized",
            "business_address",
            "business_address_normalized",
            "country",
            "country_normalized",
        ]
    ].head(20)
)

,entity_id,business_name,business_name_normalized,business_address,business_address_normalized,country,country_normalized
0,S1-925783039,Orelee's Barbershop,orelee s barbershop,"1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc,US,us
1,S1-773889195,Prime Money,prime money,"17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok,US,us
2,S1-377745466,B+ Retail Inc,b retail inc,"1712 Montebello Avenue, Phoenix, AZ",1712 montebello avenue phoenix az,US,us
3,S1-133037285,Christ Chapel,christ chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",2100 cameron drive unit apartment g dundalk md,US,us
4,S1-755362802,Prabhav Business Center,prabhav business center,"797, Lake Town Block A, Kolkata, Howrah, West ...",797 lake town block a kolkata howrah west bengal,India,india
5,S1-851869949,Custom Wealth Services LLC,custom wealth services llc,"OH, Columbus, 5559 Orville Avenue",oh columbus 5559 orville avenue,US,us
6,S1-785847572,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing private limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",2505 tower 1 oakwood runwal greens mulund gore...,India,india
7,S1-27541239,Nexus Anchor Rain,nexus anchor rain,"1111 Church Street, Unit 2007, Nashville, TN",1111 church street unit 2007 nashville tn,US,us
8,S1-629417405,Moore Bitwise Inc,moore bitwise inc,"337 Oakland Avenue, Michigan City, IN",337 oakland avenue michigan city in,US,us
9,S1-22305073,Dermatology Green Medicine,dermatology green medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",294 meadowcreek drive unit unit 2 village of p...,US,us


In [9]:
name_changed = (
    sample["business_name"]
    != sample["business_name_normalized"]
)

address_changed = (
    sample["business_address"]
    != sample["business_address_normalized"]
)

country_changed = (
    sample["country"]
    != sample["country_normalized"]
)

print("Business names changed :", name_changed.sum())
print("Addresses changed      :", address_changed.sum())
print("Countries changed      :", country_changed.sum())

Business names changed : 10000
Addresses changed      : 10000
Countries changed      : 10000


In [10]:
print(
    f"\nName change rate    : {name_changed.mean() * 100:.2f}%"
)

print(
    f"Address change rate : {address_changed.mean() * 100:.2f}%"
)

print(
    f"Country change rate  : {country_changed.mean() * 100:.2f}%"
)


Name change rate    : 100.00%
Address change rate : 100.00%
Country change rate  : 100.00%


In [11]:
start_time = time.time()

sample["business_name_script"] = (
    sample["business_name"]
    .progress_apply(detect_script)
)

elapsed = time.time() - start_time

print(f"\n✓ Script detection completed in {elapsed:.2f} seconds")

100%|██████████| 10000/10000 [00:00<00:00, 173477.49it/s]


✓ Script detection completed in 0.06 seconds


In [12]:
print(
    sample["business_name_script"]
    .value_counts()
)

business_name_script
Latin    10000
Name: count, dtype: int64


In [13]:
for script in sample["business_name_script"].unique():

    print(f"\n{'=' * 70}")
    print(f"SCRIPT: {script}")
    print("=" * 70)

    display(
        sample.loc[
            sample["business_name_script"] == script,
            [
                "entity_id",
                "business_name",
                "business_name_normalized",
                "country",
            ],
        ].head(10)
    )


SCRIPT: Latin


,entity_id,business_name,business_name_normalized,country
0,S1-925783039,Orelee's Barbershop,orelee s barbershop,US
1,S1-773889195,Prime Money,prime money,US
2,S1-377745466,B+ Retail Inc,b retail inc,US
3,S1-133037285,Christ Chapel,christ chapel,US
4,S1-755362802,Prabhav Business Center,prabhav business center,India
5,S1-851869949,Custom Wealth Services LLC,custom wealth services llc,US
6,S1-785847572,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing private limited,India
7,S1-27541239,Nexus Anchor Rain,nexus anchor rain,US
8,S1-629417405,Moore Bitwise Inc,moore bitwise inc,US
9,S1-22305073,Dermatology Green Medicine,dermatology green medicine,US


In [14]:
collision_check = (
    sample
    .groupby("business_name_normalized")
    .agg(
        original_name_count=("business_name", "nunique"),
        entity_count=("entity_id", "nunique"),
        record_count=("entity_id", "size"),
    )
    .reset_index()
)

collision_check = collision_check.sort_values(
    ["entity_count", "record_count"],
    ascending=False,
)

display(
    collision_check.head(30)
)

,business_name_normalized,original_name_count,entity_count,record_count
8290,stag llc,1,4,4
73,a cure 4 all,1,3,3
1641,chiropractic group,1,3,3
1647,chiropractic medicine,1,3,3
2898,eye group,1,3,3
5003,life projects private limited,1,3,3
6140,office of aging,1,3,3
6864,primary care group,1,3,3
8309,star global private limited,1,3,3
9133,urgent care medicine,1,3,3


In [15]:
collision_names = collision_check.loc[
    collision_check["entity_count"] > 1,
    "business_name_normalized"
].head(20)

for name in collision_names:

    print(f"\nNormalized: {name}")

    display(
        sample.loc[
            sample["business_name_normalized"] == name,
            [
                "entity_id",
                "business_name",
                "business_address",
                "country",
            ]
        ].head(20)
    )


Normalized: stag llc


,entity_id,business_name,business_address,country
2687,S1-37290366,Stag LLC,"10729 Lafayette Avenue, Shirley, IL",US
5120,S1-759182044,Stag LLC,"1200 Somerby Drive, Unit 529, Mobile, AL",US
7872,S1-687982489,Stag LLC,"New Carlisle, 207 Clay Street, OH",US
9853,S1-653948096,Stag LLC,"9005 Spring Grove, Mobile, AL",US



Normalized: a cure 4 all


,entity_id,business_name,business_address,country
5206,S1-477149670,A Cure 4 All,"3091 30, Booneville, KY",US
5570,S1-379538520,A Cure 4 All,"209 Chestnut Street, Walhalla, SC",US
8272,S1-771465672,A Cure 4 All,"AZ, Salt River Indian Res, 9730 Palm Lane",US



Normalized: chiropractic group


...

,entity_id,business_name,business_address,country
6530,S1-175294249,Beth Emanuel,"AZ, 7479 Crimson Sky Drive, Marana",US
9430,S1-218000907,Beth Emanuel,"CA, Redding, 205 Woodhill Drive",US



Normalized: bharat hospitality pvt ltd


,entity_id,business_name,business_address,country
2328,S1-422006769,Bharat Hospitality Pvt Ltd,"Flat No.5, Gulmohor Orchid S.No2937, Sahani Su...",India
9199,S1-733133405,Bharat Hospitality Pvt Ltd,"766, Anna Salai, Sakthi Towers-1, Chennai, Tam...",India



Normalized: bharat industries private limited


,entity_id,business_name,business_address,country
1771,S1-881015038,Bharat Industries Private Limited,"B-3/276, Chitrakoot Vaishali Nagar, Jaipur, Ra...",India
2496,S1-585596917,Bharat Industries Private Limited,"13-04-16/1, Flat No.106, Ground Floor Gayathri...",India


In [16]:
benchmark_sizes = [1_000, 5_000, 10_000]

benchmark_results = []

for size in benchmark_sizes:

    test_sample = (
        train_data["source1"]
        .head(size)
        .copy()
    )

    start = time.time()

    test_sample["business_name_normalized"] = (
        test_sample["business_name"]
        .progress_apply(normalize_business_name)
    )

    elapsed = time.time() - start

    benchmark_results.append({
        "rows": size,
        "seconds": elapsed,
        "rows_per_second": size / elapsed,
    })

benchmark_df = pd.DataFrame(benchmark_results)

display(benchmark_df)

100%|██████████| 10000/10000 [00:00<00:00, 77394.39it/s]


,rows,seconds,rows_per_second
0,1000,0.041558,24062.739878
1,5000,0.083798,59667.343815
2,10000,0.135528,73785.572923


<!-- measured-pipeline-supplement -->
# Unicode correction and representations

Reproducible views of measured CLI artifacts. Historical cells above are preserved; do not rerun the original full-memory blocking experiment on a memory-limited machine. New cells are supplied unexecuted.

In [ ]:
from pathlib import Path
import json, sys
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.normalization import representations
[representations(s) for s in ['राम मार्केटिंग', 'Café', 'Москва', '北京']]